# T60 AUV 训练

训练行为全部来自版本化 recipe。Notebook 只选择 recipe、运行名、seed、环境数和启动开关；实际输入会在 worker 创建 run 后写入该 run 的 `params/inputs/`。

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'simulation/training').is_dir():
    raise RuntimeError('请从 isaac-auv-env 仓库根目录启动 train.ipynb。')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from simulation.training import (
    build_training_campaign, follow_campaign_log, launch_or_attach_campaign,
)
from simulation.training.campaign import build_train_command, display_command


## 1. 选择运行

In [ ]:
RECIPE_PATH = REPO_ROOT / 'simulation/training/recipes/t60_trajectory_policy_6_v1.json'
RUN_NAME = 't60_policy_6'
SEED = 42
NUM_ENVS = 2048
HEADLESS = True
START_TRAINING = True


## 2. 校验 recipe 并预览命令

In [3]:
campaign = build_training_campaign(
    isaaclab_root=Path.home() / 'IsaacLab',
    recipe_path=RECIPE_PATH,
    seed=SEED,
    num_envs=NUM_ENVS,
    run_name=RUN_NAME,
    headless=HEADLESS,
)
print(f'recipe: {campaign.recipe.name}')
print(f'architecture: {campaign.recipe.mlp_architecture}')
print(f'reward: {campaign.recipe.reward_profile}')
print(display_command(
    build_train_command(campaign.experiment, campaign.train),
    cwd=campaign.experiment.isaaclab_root,
))


recipe: t60-trajectory-policy6-v1
architecture: mlp_history_5
reward: policy_6
./isaaclab.sh -p source/isaaclab_tasks/isaaclab_tasks/direct/isaac-auv-env/simulation/training/train.py --task Isaac-AUV-Traj-Direct-v1 --num_envs 1024 --seed 42 --training_recipe ../isaac-auv-env/simulation/training/recipes/t60_trajectory_policy_6_v1.json 'env.mlp_architecture="mlp_history_5"' 'agent.experiment_name="/home/jining_yang/isaac-auv-env/simulation/rlpolicy/auv_traj_mlp_history_5"' 'agent.policy.actor_hidden_dims=[512,384,256,128]' 'agent.policy.critic_hidden_dims=[512,384,256,128]' --max_iterations 500 --run_name t60_policy_6 --headless --logger tensorboard agent.num_steps_per_env=256 env.tracking_reward_profile=policy_6 env.trajectory_curriculum=true 'env.trajectory_amp_x_range=[0.6,0.78]' 'env.trajectory_amp_y_range=[0.55,0.75]' 'env.trajectory_amp_z_range=[0.08,0.2]' 'env.trajectory_period_range=[10.0,20.0]' 'env.trajectory_speed_levels_mps=[0.1,0.2,0.3,0.4]' 'env.trajectory_curriculum_stage_

## 3. 启动或连接训练

仅当 `START_TRAINING=True` 时启动。中断日志显示不会终止后台 worker。

In [4]:
if not START_TRAINING:
    print('未启动：将 START_TRAINING 改为 True 后重新运行本单元。')
else:
    pid, log_path, started = launch_or_attach_campaign(campaign)
    message = '训练已启动' if started else '已连接正在运行的训练'
    print(f'{message}\nPID: {pid}\nlog: {log_path}')
    print(f'TensorBoard: tensorboard --logdir {campaign.experiment.logs_root}')
    follow_campaign_log(pid, log_path, from_start=started)


未启动：将 START_TRAINING 改为 True 后重新运行本单元。
